# 📊 Real Estate Dataset — Analysis & Insights
**Dataset:** `feature_engineered_dataset.csv`  
**Author:** SidduReddyy  
**Date:** 2026-03-13

This notebook performs post-feature-engineering analysis including:
- Neighbourhood-level summary statistics
- Yearly price trend analysis
- Property type breakdown
- Overvalued vs. Undervalued property classification
- Monthly seasonality analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load the feature-engineered dataset
df = pd.read_csv('feature_engineered_dataset.csv')
print(f'Dataset Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(5)

## 1. Basic Dataset Overview

In [ ]:
# Basic statistics
print('=== Dataset Info ===')
print(f'Total Records     : {len(df)}')
print(f'Total Columns     : {len(df.columns)}')
print(f'Missing Values    : {df.isnull().sum().sum()}')
print(f'Sale Year Range   : {df["SaleYear"].min()} - {df["SaleYear"].max()}')
print(f'Unique Neighborhoods: {df["xrPrimaryNeighborhoodID"].nunique()}')
print(f'Avg Sale Price    : ${df["SalePrice"].mean():,.2f}')
print(f'Median Sale Price : ${df["SalePrice"].median():,.2f}')
print(f'Min Sale Price    : ${df["SalePrice"].min():,.2f}')
print(f'Max Sale Price    : ${df["SalePrice"].max():,.2f}')

df.describe()

## 2. Neighbourhood-Level Summary Statistics

In [ ]:
neighborhood_summary = df.groupby('xrPrimaryNeighborhoodID').agg(
    Total_Sales        = ('SalePrice', 'count'),
    Avg_SalePrice      = ('SalePrice', 'mean'),
    Median_SalePrice   = ('SalePrice', 'median'),
    Min_SalePrice      = ('SalePrice', 'min'),
    Max_SalePrice      = ('SalePrice', 'max'),
    Avg_PricePerSqft   = ('PricePerSqft', 'mean'),
    Avg_AppraisalRatio = ('AppraisalRatio', 'mean'),
    Avg_LandSF         = ('LandSF', 'mean'),
    Avg_FinishedArea   = ('TotalFinishedArea', 'mean'),
).reset_index().round(2)

print(f'Total Neighborhoods analyzed: {len(neighborhood_summary)}')

# Top 10 neighborhoods by average sale price
top10 = neighborhood_summary.nlargest(10, 'Avg_SalePrice')[['xrPrimaryNeighborhoodID','Avg_SalePrice','Avg_PricePerSqft','Total_Sales']]
print('\nTop 10 Neighborhoods by Avg Sale Price:')
print(top10.to_string(index=False))

# Save to CSV
neighborhood_summary.to_csv('neighborhood_summary.csv', index=False)
print('\nneighborhood_summary.csv saved!')

neighborhood_summary.head(10)

In [ ]:
# Plot: Top 10 neighborhoods by Avg Sale Price
top10_plot = neighborhood_summary.nlargest(10, 'Avg_SalePrice')

plt.figure(figsize=(12, 5))
plt.bar(top10_plot['xrPrimaryNeighborhoodID'].astype(str), top10_plot['Avg_SalePrice'], color='steelblue')
plt.title('Top 10 Neighborhoods by Average Sale Price', fontsize=14)
plt.xlabel('Neighborhood ID')
plt.ylabel('Avg Sale Price ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Yearly Price Trend Analysis

In [ ]:
yearly_trends = df.groupby('SaleYear').agg(
    Total_Transactions  = ('SalePrice', 'count'),
    Avg_SalePrice       = ('SalePrice', 'mean'),
    Median_SalePrice    = ('SalePrice', 'median'),
    Total_Sales_Volume  = ('SalePrice', 'sum'),
    Avg_PricePerSqft    = ('PricePerSqft', 'mean'),
    Avg_AppraisalRatio  = ('AppraisalRatio', 'mean'),
).reset_index().round(2)

yearly_trends.to_csv('yearly_price_trends.csv', index=False)
print('yearly_price_trends.csv saved!')
print(yearly_trends.to_string(index=False))

In [ ]:
# Plot: Yearly Avg Sale Price Trend
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(yearly_trends['SaleYear'], yearly_trends['Avg_SalePrice'], 'b-o', label='Avg Sale Price')
ax1.plot(yearly_trends['SaleYear'], yearly_trends['Median_SalePrice'], 'g--s', label='Median Sale Price')
ax1.set_xlabel('Year')
ax1.set_ylabel('Price ($)')
ax1.set_title('Yearly Sale Price Trends', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Overvalued vs. Undervalued Property Classification

Based on **Appraisal Ratio**:
- `> 1.5` → **Overvalued** (Sales price much higher than appraisal)
- `< 0.7` → **Undervalued** (Sales price much lower than appraisal)
- Otherwise → **Fair**

In [ ]:
df['ValuationCategory'] = 'Fair'
df.loc[df['AppraisalRatio'] > 1.5, 'ValuationCategory'] = 'Overvalued'
df.loc[df['AppraisalRatio'] < 0.7, 'ValuationCategory'] = 'Undervalued'

valuation_counts = df['ValuationCategory'].value_counts().reset_index()
valuation_counts.columns = ['ValuationCategory', 'Count']
valuation_counts['Percentage'] = (valuation_counts['Count'] / len(df) * 100).round(2)

valuation_counts.to_csv('valuation_category_summary.csv', index=False)
print('valuation_category_summary.csv saved!')
print(valuation_counts.to_string(index=False))

In [ ]:
# Pie chart
plt.figure(figsize=(7, 7))
colors = ['#2ecc71', '#e74c3c', '#3498db']
plt.pie(valuation_counts['Count'], labels=valuation_counts['ValuationCategory'],
        autopct='%1.1f%%', colors=colors, startangle=140)
plt.title('Property Valuation Categories', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Monthly Seasonality Analysis

In [ ]:
monthly_trends = df.groupby('SaleMonth').agg(
    Total_Transactions = ('SalePrice', 'count'),
    Avg_SalePrice      = ('SalePrice', 'mean'),
    Avg_PricePerSqft   = ('PricePerSqft', 'mean'),
).reset_index()

month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
monthly_trends['MonthName'] = monthly_trends['SaleMonth'].map(month_names)
monthly_trends = monthly_trends.round(2)

monthly_trends.to_csv('monthly_seasonality.csv', index=False)
print('monthly_seasonality.csv saved!')
print(monthly_trends[['MonthName','Total_Transactions','Avg_SalePrice']].to_string(index=False))

In [ ]:
# Bar chart: transactions by month
plt.figure(figsize=(12, 5))
plt.bar(monthly_trends['MonthName'], monthly_trends['Total_Transactions'], color='coral')
plt.title('Number of Sales by Month (Seasonality)', fontsize=14)
plt.xlabel('Month')
plt.ylabel('Number of Transactions')
plt.tight_layout()
plt.show()

## 6. Price Per SqFt Distribution

In [ ]:
# Filter out extreme outliers for cleaner visualization
df_clean = df[(df['PricePerSqft'] > 0) & (df['PricePerSqft'] < 600)]

plt.figure(figsize=(12, 5))
plt.hist(df_clean['PricePerSqft'], bins=50, color='steelblue', edgecolor='white')
plt.axvline(df_clean['PricePerSqft'].mean(), color='red', linestyle='--', label=f'Mean: ${df_clean["PricePerSqft"].mean():.1f}')
plt.axvline(df_clean['PricePerSqft'].median(), color='orange', linestyle='--', label=f'Median: ${df_clean["PricePerSqft"].median():.1f}')
plt.title('Price Per SqFt Distribution', fontsize=14)
plt.xlabel('Price per SqFt ($)')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Mean  Price/SqFt: ${df_clean["PricePerSqft"].mean():.2f}')
print(f'Median Price/SqFt: ${df_clean["PricePerSqft"].median():.2f}')
print(f'Std   Price/SqFt: ${df_clean["PricePerSqft"].std():.2f}')

## ✅ Summary

| Analysis | Output File |
|---|---|
| Neighborhood Summary | `neighborhood_summary.csv` |
| Yearly Price Trends | `yearly_price_trends.csv` |
| Valuation Categories | `valuation_category_summary.csv` |
| Monthly Seasonality | `monthly_seasonality.csv` |

All analysis complete! 🎉